# Weight quantization
What is lost when linear weights use INT8?

In [ ]:
import os
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(root / 'src'))
import torch
torch.set_num_threads(1)
torch.manual_seed(42)
smoke = os.environ.get('PYTHONIC_SMOKE') == '1'

In [ ]:
from pythonic.model import Decoder, DecoderConfig
from pythonic.quantization import quantize_linears, storage_bytes
from pythonic.experiments import benchmark
model = Decoder(DecoderConfig(width=32, layers=2, context=64)).eval()
quantized = quantize_linears(model)
tokens = torch.randint(0, 258, (1, 32))
with torch.inference_mode():
    print('logit error:', (model(tokens) - quantized(tokens)).abs().max().item())
    for name, candidate in [('float', model), ('int8', quantized)]:
        timing = benchmark(lambda: candidate(tokens), torch.device('cpu'), repeats=3 if smoke else 20)
        print(name, storage_bytes(candidate), 'bytes;', round(timing.median_ms, 3), 'ms')

In [ ]:
if not smoke:
    from pythonic.data import load_papers, split_papers
    from pythonic.training import load_decoder, validation_loss
    checkpoint = root / 'artifacts/decoder/decoder.pt'
    if checkpoint.exists():
        trained = load_decoder(checkpoint)
        compressed = quantize_linears(trained)
        split = split_papers(load_papers(root / 'data/pubmedqa.json'))
        print('float validation loss:', validation_loss(trained, split.validation))
        print('int8 validation loss:', validation_loss(compressed, split.validation))

Weights are dequantized for each call. This measures storage and numerical changes, not a specialized INT8 kernel.